# 📦 Data Lake Ingestion

Download datasets to Google Drive using Colab's high-speed network.

**Benefits:**
- 🚀 Fast downloads (Google's network)
- 💾 Direct to Drive (no local storage needed)
- 📁 Auto-organized by category (vision, nlp, medical, etc.)

---
## 1️⃣ Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [21]:
!pip install -q ucimlrepo


🎓 Education Dataset Downloader
   20 datasets → /content/drive/MyDrive/data_lake/01_bronze_education

[1/20] student-performance (1MB) via uci
  ⏭️  Already exists, skipping

[2/20] student-academics (500KB) via uci
  ⏭️  Already exists, skipping

[3/20] higher-education-dropout (2MB) via uci
  ⏭️  Already exists, skipping

[4/20] national-poll-happiness (100KB) via uci
  ❌ Error: Dataset with ID 826 not found

[5/20] open-university-learning (5MB) via uci
  ❌ Error: "Open University Learning Analytics dataset" dataset (id=349) exists in the repository, but is not available for import. Please select a dataset from this list: https://archive.ics.uci.edu/datasets?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true

[6/20] students-exam-scores (5MB) via kaggle
  ✅ Saved to /content/drive/MyDrive/data_lake/01_bronze_education/students-exam-scores

[7/20] student-study-performance (1MB) via kaggle
  ✅ Saved to /content/drive/MyDrive/data_lake/01_bronze_education/student-study-perfo

Found 20 education datasets
📦 Estimated download: 0.2GB
ℹ️  Saving to Google Drive - space check skipped (uses Drive quota)

📦 Downloading 20 datasets...

[20/20] us-college-scorecard: 100%|██████████| 20/20 [00:41<00:00,  2.10s/dataset]
  ⚠️  Attempt 1/3 failed: Dataset with ID 826 not found
     Retrying in 5s...
  ⚠️  Attempt 2/3 failed: Dataset with ID 826 not found
     Retrying in 10s...
  ❌ All 3 attempts failed: Dataset with ID 826 not found
❌ national-poll-happiness download failed
⏭️  student-academics already exists in education, skipping...
  ⚠️  Dataset access denied: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset
❌ literacy-rates-world download failed
⏭️  school-performance already exists in education, skipping...
⏭️  student-performance already exists in education, skipping...
⏭️  student-study-performance already exists in education, skipping...
  ⚠️  Dataset access denied: 403 Client Error: Forbidden for url: https:

In [3]:
# Copy API tokens from Drive
SECRETS = "/content/drive/MyDrive/.secrets"

!mkdir -p ~/.kaggle ~/.secrets
!cp "{SECRETS}/kaggle.json" ~/.kaggle/ 2>/dev/null && chmod 600 ~/.kaggle/kaggle.json && echo "✅ Kaggle" || echo "❌ Kaggle"
!cp "{SECRETS}/huggingface_token" ~/.secrets/ 2>/dev/null && echo "✅ HuggingFace" || echo "⚠️ HuggingFace (optional)"

✅ Kaggle
✅ HuggingFace


In [4]:
# Setup paths and shared config
import sys



/content/drive/MyDrive/repos/data-ingestion


---
## 2️⃣ Browse Datasets

In [5]:
# List all available datasets
# !python scripts/ingest.py --list

# Filter by category
# !python scripts/ingest.py --list vision
# !python scripts/ingest.py --list medical
# !python scripts/ingest.py --list nlp

---
## 3️⃣ Download Datasets

In [6]:
# Download a single dataset
# DATASET = "chest-xray"  # @param {type:"string"}
# !python scripts/ingest.py {DATASET}

In [7]:
# Batch download — top priority datasets
# !python scripts/batch_download.py --priority

# Batch download — by category
# !python scripts/batch_download.py --category vision
# !python scripts/batch_download.py --category nlp
# !python scripts/batch_download.py --category tabular
# !python scripts/batch_download.py --category audio
# !python scripts/batch_download.py --category timeseries
# !python scripts/batch_download.py --category education

# Parallel downloads
# !python scripts/batch_download.py --priority --parallel 4

---
## 4️⃣ 🏥 Medical Imaging Datasets

Download medical imaging datasets by modality:
- 🔊 **Ultrasound** — 🔬 **CT Scan** — 🧲 **MRI** — ☢️ **X-ray**

In [8]:
# Download ALL medical datasets
# !python scripts/batch_download.py --medical

# Download by imaging modality
# !python scripts/batch_download.py --ultrasound  # 20+ ultrasound datasets
# !python scripts/batch_download.py --xray        # X-ray datasets
# !python scripts/batch_download.py --ct          # CT scan datasets
# !python scripts/batch_download.py --mri         # MRI datasets

---
## 5️⃣ 🔬 Polyp Detection Dataset

Downloads **Kvasir-SEG** (1,000 images) + **CVC-ClinicDB** (612 images),
converts masks → **YOLO bounding boxes**, creates train/val/test split.

In [9]:
!pip install -q opencv-python-headless

In [10]:
# Download Kvasir-SEG (official Simula URL)
import os, zipfile, shutil
from pathlib import Path

WORK_DIR = Path('/content/polyp_data')
WORK_DIR.mkdir(exist_ok=True)

kvasir_dir = WORK_DIR / 'Kvasir-SEG'
if not kvasir_dir.exists():
    kvasir_zip = WORK_DIR / 'Kvasir-SEG.zip'
    if not kvasir_zip.exists() or kvasir_zip.stat().st_size < 1_000_000:
        print('Downloading Kvasir-SEG from simula.no...')
        !wget -q --show-progress --no-check-certificate -O {kvasir_zip} 'https://datasets.simula.no/kvasir-seg/Kvasir-SEG.zip'
    fsize = kvasir_zip.stat().st_size if kvasir_zip.exists() else 0
    print(f'Downloaded: {fsize / 1e6:.1f} MB')
    if fsize > 1_000_000:
        !cd {WORK_DIR} && unzip -q -o Kvasir-SEG.zip
        if not kvasir_dir.exists():
            for d in WORK_DIR.iterdir():
                if d.is_dir() and (d / 'images').exists() and (d / 'masks').exists():
                    d.rename(kvasir_dir)
                    break
        print('✅ Kvasir-SEG extracted')
    else:
        print('❌ Download failed')
else:
    print('Kvasir-SEG already exists')

if kvasir_dir.exists():
    for p in sorted(kvasir_dir.iterdir()):
        if p.is_dir():
            print(f'  {p.name}/  ({len(list(p.iterdir()))} files)')

Downloaded: 0.0 MB
❌ Download failed


In [11]:
# Download CVC-ClinicDB via Kaggle
cvc_dir = WORK_DIR / 'CVC-ClinicDB'
if not cvc_dir.exists():
    print('Downloading CVC-ClinicDB from Kaggle...')
    !kaggle datasets download -d balrajashwath/cvcclinicdb -p {WORK_DIR} --unzip
    # Find the extracted directory
    if not cvc_dir.exists():
        for d in WORK_DIR.iterdir():
            if d.is_dir() and d.name != 'Kvasir-SEG':
                if any(d.rglob('*.png')) or any(d.rglob('*.tif')):
                    d.rename(cvc_dir)
                    break
        # If files extracted flat in polyp_data, organize them
        if not cvc_dir.exists():
            png_files = list(WORK_DIR.glob('*.png')) + list(WORK_DIR.glob('*.tif'))
            if png_files:
                cvc_dir.mkdir()
                for f in png_files:
                    shutil.move(str(f), str(cvc_dir / f.name))
    print('✅ CVC-ClinicDB extracted')
else:
    print('CVC-ClinicDB already exists')

# Show what we have
for p in sorted(WORK_DIR.iterdir()):
    if p.is_dir():
        imgs = len(list(p.rglob('*.png'))) + len(list(p.rglob('*.jpg'))) + len(list(p.rglob('*.tif')))
        print(f'  {p.name}/  ({imgs} images)')

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
✅ CVC-ClinicDB extracted


In [12]:
# Convert segmentation masks to YOLO bounding boxes
import cv2
import numpy as np

def mask_to_yolo_bbox(mask_path):
    """Convert mask → YOLO bounding boxes. Merges nearby polyps."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    h, w = mask.shape
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 100:
            continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        boxes.append([x, y, x + bw, y + bh])
    if not boxes:
        return []
    # Merge nearby boxes (within 50px)
    merged = list(boxes)
    changed = True
    while changed:
        changed = False
        new_merged, used = [], set()
        for i in range(len(merged)):
            if i in used: continue
            cur = list(merged[i])
            for j in range(i+1, len(merged)):
                if j in used: continue
                o = merged[j]
                if cur[0]-50<=o[2] and cur[2]+50>=o[0] and cur[1]-50<=o[3] and cur[3]+50>=o[1]:
                    cur = [min(cur[0],o[0]),min(cur[1],o[1]),max(cur[2],o[2]),max(cur[3],o[3])]
                    used.add(j); changed = True
            new_merged.append(cur); used.add(i)
        merged = new_merged
    return [[0, ((x1+x2)/2)/w, ((y1+y2)/2)/h, (x2-x1)/w, (y2-y1)/h] for x1,y1,x2,y2 in merged]

# Quick test
test_masks = list((kvasir_dir / 'masks').glob('*.jpg')) or list((kvasir_dir / 'masks').glob('*.png'))
if test_masks:
    r = mask_to_yolo_bbox(test_masks[0])
    print(f'Test: {test_masks[0].name} → {len(r)} box(es)')

In [13]:
# Collect image-mask pairs from both datasets
import random
random.seed(42)

YOLO_DIR = BRONZE_DETECTION / 'polyp_yolo'
for split in ['train', 'val', 'test']:
    (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def find_pairs(name, img_dir, mask_dir):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif'}
    imgs = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in exts])
    ml = {m.stem: m for m in mask_dir.iterdir() if m.suffix.lower() in exts}
    pairs = [(name, i, ml[i.stem]) for i in imgs if i.stem in ml]
    print(f'  {name}: {len(pairs)} pairs'); return pairs

all_pairs = []
# Kvasir-SEG
if (kvasir_dir / 'images').exists():
    all_pairs.extend(find_pairs('kvasir', kvasir_dir / 'images', kvasir_dir / 'masks'))
# CVC-ClinicDB — auto-detect
cvc_found = False
for in_, mn_ in [('Original','Ground Truth'),('images','masks')]:
    for base in [cvc_dir, WORK_DIR / 'CVC-612']:
        if base.exists() and (base/in_).exists() and (base/mn_).exists():
            all_pairs.extend(find_pairs('cvc', base/in_, base/mn_))
            cvc_found = True; break
    if cvc_found: break
if not cvc_found:
    for d in WORK_DIR.rglob('*'):
        if d.is_dir() and d.name.lower() in ('original','images'):
            for mn in ('Ground Truth','masks','GT'):
                if (d.parent/mn).exists():
                    all_pairs.extend(find_pairs('cvc', d, d.parent/mn))
                    cvc_found = True; break
        if cvc_found: break
print(f'\nTotal: {len(all_pairs)} pairs')


Total: 0 pairs


In [14]:
# Split and convert to YOLO format (80/10/10)
random.shuffle(all_pairs)
n = len(all_pairs)
nt, nv = int(0.8*n), int(0.1*n)
splits = {'train': all_pairs[:nt], 'val': all_pairs[nt:nt+nv], 'test': all_pairs[nt+nv:]}
stats, skipped = {'train':0,'val':0,'test':0}, 0

for sn, pairs in splits.items():
    for ds, ip, mp in pairs:
        labels = mask_to_yolo_bbox(mp)
        if not labels: skipped += 1; continue
        un = f'{ds}_{ip.stem}'
        di = YOLO_DIR/'images'/sn/f'{un}.jpg'
        if not di.exists():
            img = cv2.imread(str(ip))
            if img is not None: cv2.imwrite(str(di), img)
        with open(YOLO_DIR/'labels'/sn/f'{un}.txt','w') as f:
            for l in labels:
                f.write(' '.join(f'{v:.6f}' if i>0 else str(int(v)) for i,v in enumerate(l))+'\n')
        stats[sn] += 1

print(f'\n✅ YOLO dataset: {YOLO_DIR}')
print(f'  Train: {stats["train"]}  Val: {stats["val"]}  Test: {stats["test"]}  Skipped: {skipped}')


✅ YOLO dataset: /content/drive/MyDrive/data_lake/01_bronze_detection/polyp_yolo
  Train: 0  Val: 0  Test: 0  Skipped: 0


In [15]:
# Write dataset.yaml for YOLOv8
import yaml
yaml_path = YOLO_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump({'path': str(YOLO_DIR), 'train': 'images/train', 'val': 'images/val',
               'test': 'images/test', 'nc': 1, 'names': ['polyp']}, f)
print(f'✅ {yaml_path}'); print(open(yaml_path).read())

✅ /content/drive/MyDrive/data_lake/01_bronze_detection/polyp_yolo/dataset.yaml
names:
- polyp
nc: 1
path: /content/drive/MyDrive/data_lake/01_bronze_detection/polyp_yolo
test: images/test
train: images/train
val: images/val



In [16]:
# Visualize 6 random training samples with bounding boxes
import matplotlib.pyplot as plt
import matplotlib.patches as patches
train_images = sorted((YOLO_DIR/'images'/'train').glob('*.jpg'))
samples = random.sample(train_images, min(6, len(train_images)))
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, ip in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]; ax.imshow(img)
    lp = YOLO_DIR/'labels'/'train'/f'{ip.stem}.txt'
    if lp.exists():
        for line in open(lp):
            _, xc, yc, bw, bh = [float(x) for x in line.split()]
            ax.add_patch(patches.Rectangle(((xc-bw/2)*w,(yc-bh/2)*h),bw*w,bh*h,lw=3,ec='lime',fc='none'))
    ax.set_title(ip.stem, fontsize=9); ax.axis('off')
plt.suptitle('Polyp Detection — YOLO Samples', fontsize=16); plt.tight_layout(); plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [17]:
# Copy custom polyp images for later inference
custom_dir = LANDING / 'polyp'
custom_dest = YOLO_DIR / 'custom_unlabeled'
custom_dest.mkdir(exist_ok=True)
if custom_dir.exists():
    imgs = list(custom_dir.glob('*.jpg')) + list(custom_dir.glob('*.png'))
    for i in imgs: shutil.copy2(i, custom_dest / i.name)
    print(f'Copied {len(imgs)} custom polyp images')
else:
    print(f'No custom polyps at {custom_dir}')
print('\n✅ Polyp data preparation complete!')

Copied 0 custom polyp images

✅ Polyp data preparation complete!


---
## 6️⃣ Verify & Audit

In [18]:
print('Data Lake:')
for d in sorted(DATA_LAKE.iterdir()):
    if d.is_dir() and d.name != '09_cache':
        ch = list(d.iterdir())
        if ch: print(f'  {d.name}/ ({len(ch)} items)')

Data Lake:
  00_landing/ (13 items)
  01_bronze_audio/ (5 items)
  01_bronze_detection/ (3 items)
  01_bronze_education/ (3 items)
  01_bronze_generative/ (1 items)
  01_bronze_medical/ (24 items)
  01_bronze_nlp/ (18 items)
  01_bronze_tabular/ (19 items)
  01_bronze_timeseries/ (5 items)
  01_bronze_vision/ (45 items)
  02_silver/ (1 items)
  06_inference/ (1 items)


In [19]:
# Audit data lake for issues (empty folders, failed downloads)
# !python scripts/audit_data_lake.py

# Fix issues: remove empty folders
# !python scripts/audit_data_lake.py --fix

# Re-download failed datasets
# !python scripts/audit_data_lake.py --redownload

In [20]:
# Check data lake disk usage
!du -sh /content/drive/MyDrive/data_lake/01_bronze_*

2.9G	/content/drive/MyDrive/data_lake/01_bronze_audio
38G	/content/drive/MyDrive/data_lake/01_bronze_detection
590K	/content/drive/MyDrive/data_lake/01_bronze_education
du: cannot read directory '/content/drive/MyDrive/data_lake/01_bronze_generative/celeba/celeba/img_align_celeba': Input/output error
1.4G	/content/drive/MyDrive/data_lake/01_bronze_generative
^C


---
## 📚 Quick Reference

### Folder Structure
```
data_lake/
├── 01_bronze_vision/     # Image datasets (CIFAR, ImageNet, etc.)
├── 01_bronze_nlp/        # Text datasets (IMDB, SQuAD, etc.)
├── 01_bronze_medical/    # Medical imaging (chest-xray, brain-tumor, etc.)
├── 01_bronze_audio/      # Audio datasets
├── 01_bronze_tabular/    # Tabular datasets (Titanic, etc.)
└── 01_bronze_timeseries/ # Time series data
```

### Batch Download Commands
| Command | Description |
|---------|-------------|
| `--priority` | Top 25 essential datasets |
| `--vision` | All vision datasets |
| `--medical` | All medical datasets (50+) |
| `--ultrasound` | Ultrasound only (20+) |
| `--xray` | X-ray only |
| `--ct` | CT scan only |
| `--mri` | MRI only |
| `--nlp` | NLP datasets |
| `--education` | Education datasets |
| `--parallel 4` | Use 4 concurrent downloads |

### Token Setup (My Drive/.secrets/)
| File | Format | Required For |
|------|--------|-------------|
| `kaggle.json` | `{"username":"...","key":"..."}` | Kaggle datasets |
| `huggingface_token` | `hf_xxxxx` | Gated datasets (ImageNet) |